# Experiment

## Import libraries

In [21]:
import pandas as pd

file_path = "iisd_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="quarter_str",
    value_name="value",
)

# Filter out any non-quarter columns
df = df[df["quarter_str"].str.match(r"Q\d+/\d{4}")]

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and quarter
df["quarter"] = df["quarter_str"].str.extract(r"Q(\d+)/")[0].astype(int)
df["year"] = df["quarter_str"].str.extract(r"/(\d{4})")[0].astype(int)

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "quarter"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and quarter
df = df.sort_values(["year", "quarter"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

Chỉ tiêu,year,quarter,foreign_direct_investment_capital,government_bond_capital,investment_capital_from_residents_and_private_individuals,investment_capital_from_the_state_budget,investment_capital_of_state_enterprises_equity,loans_from_other_sources_of_the_state_sector,other_mobilized_capital,planned_state_investment_credit,total
0,2014,1,58.90,9.03,77.50,34.40,11.20,11.00,2.30,10.50,351.23
1,2014,3,65.20,17.70,136.20,56.28,16.50,21.80,5.10,14.30,456.78
2,2014,4,76.50,20.00,152.30,60.60,20.10,29.10,9.20,16.40,384.20
3,2015,1,67.20,10.20,89.70,37.00,12.30,15.40,3.20,11.10,246.10
4,2015,2,69.70,15.90,108.60,55.62,15.90,18.10,4.50,14.30,302.62
5,2015,3,76.60,16.70,151.80,59.73,17.30,21.80,5.30,18.60,367.83
6,2015,4,104.60,18.90,179.50,68.06,21.40,31.50,8.20,18.50,450.66
7,2016,1,76.50,8.80,102.50,42.80,13.10,17.20,3.50,12.10,276.50
8,2016,2,80.70,11.20,123.70,64.36,16.20,19.50,4.30,15.40,335.36
9,2016,3,78.30,6.10,161.50,73.14,19.20,24.00,5.40,18.90,386.54


In [22]:
df.columns

Index(['year', 'quarter', 'foreign_direct_investment_capital',
       'government_bond_capital',
       'investment_capital_from_residents_and_private_individuals',
       'investment_capital_from_the_state_budget',
       'investment_capital_of_state_enterprises_equity',
       'loans_from_other_sources_of_the_state_sector',
       'other_mobilized_capital', 'planned_state_investment_credit', 'total'],
      dtype='object', name='Chỉ tiêu')